# Internship Assignment - Beginner Walkthrough

This notebook shows the complete logic in one place:

1. Read the four CSV files
2. Clean duplicate and missing data
3. Join the tables
4. Calculate revenue, IR%, and ISU%
5. Answer all 10 questions

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('datasets')

## Step 1 - Read the four CSV files

In [ ]:
campaigns = pd.read_csv(DATA_DIR / 'dim_campaigns.csv')
products = pd.read_csv(DATA_DIR / 'dim_products.csv')
stores = pd.read_csv(DATA_DIR / 'dim_stores.csv')
events_original = pd.read_csv(DATA_DIR / 'fact_events.csv')

print('Campaign rows:', len(campaigns))
print('Product rows:', len(products))
print('Store rows:', len(stores))
print('Event rows:', len(events_original))
events_original.head()

## Step 2 - Remove duplicates

The client defines a duplicate using `store_id`, `campaign_id`, and `product_code`. We keep the first matching row.

In [ ]:
duplicate_key = ['store_id', 'campaign_id', 'product_code']
events = events_original.drop_duplicates(subset=duplicate_key, keep='first').copy()
duplicates_removed = len(events_original) - len(events)
print('Answer 1 - Duplicate rows removed:', duplicates_removed)

## Step 3 - Fill missing pre-promotion quantities

First count the missing values. Then calculate the median from the cleaned data and use it to fill them.

In [ ]:
quantity_before = 'quantity_sold(before_promo)'
missing_count = events[quantity_before].isna().sum()
median_quantity = events[quantity_before].median()
events[quantity_before] = events[quantity_before].fillna(median_quantity)
print('Answer 3 - Missing values and median:', missing_count, median_quantity)

## Step 4 - Join the tables

The events table contains IDs. Joining adds campaign names, product names, categories, and cities.

In [ ]:
data = events.merge(campaigns, on='campaign_id', validate='many_to_one')
data = data.merge(products, on='product_code', validate='many_to_one')
data = data.merge(stores, on='store_id', validate='many_to_one')
print('Rows after joining:', len(data))
data.head()

## Step 5 - Calculate revenue

Revenue = price multiplied by quantity. The supplied after-promotion price and quantity are used directly, including for BOGOF.

In [ ]:
price_before = 'base_price(before_promo)'
quantity_after = 'quantity_sold(after_promo)'
price_after = 'base_price(after_promo)'

data['revenue_before'] = data[price_before] * data[quantity_before]
data['revenue_after'] = data[price_after] * data[quantity_after]
data[[price_before, quantity_before, 'revenue_before', price_after, quantity_after, 'revenue_after']].head()

## Step 6 - Helper for IR% and ISU%

For a product, store, campaign, or promotion, we first add its totals. Then:

`IR% = (revenue after - revenue before) / revenue before * 100`

`ISU% = (units after - units before) / units before * 100`

In [ ]:
def calculate_metrics(filtered_data, group_columns):
    summary = filtered_data.groupby(group_columns, as_index=False)[
        ['revenue_before', 'revenue_after', quantity_before, quantity_after]
    ].sum()
    summary['IR%'] = (
        (summary['revenue_after'] - summary['revenue_before'])
        / summary['revenue_before'] * 100
    )
    summary['ISU%'] = (
        (summary[quantity_after] - summary[quantity_before])
        / summary[quantity_before] * 100
    )
    return summary

## Question 1 - How many duplicate rows were removed?

In [ ]:
print('Answer:', duplicates_removed)  # 10

## Question 2 - How many cities have more than 5 stores?

In [ ]:
store_count_by_city = stores.groupby('city').size().sort_values(ascending=False)
cities_more_than_5 = store_count_by_city[store_count_by_city > 5]
print('Answer:', len(cities_more_than_5))
cities_more_than_5

## Question 3 - How many missing values were filled, and what median was used?

In [ ]:
print(f'Answer: {missing_count},{median_quantity:g}')  # 20,78

## Question 4 - Which category had the lowest average price before promotion?

In [ ]:
category_prices = data.groupby('category')[price_before].mean().sort_values()
print('Answer:', category_prices.index[0], round(category_prices.iloc[0], 2))
category_prices

## Question 5 - Total quantity sold after BOGOF during Diwali

In [ ]:
diwali_bogof = data[(data['campaign_name'] == 'Diwali') & (data['promo_type'] == 'BOGOF')]
diwali_bogof_units = diwali_bogof[quantity_after].sum()
print('Answer:', diwali_bogof_units)  # 34461

## Question 6 - Which store sold the most units after promotion during Diwali?

In [ ]:
diwali = data[data['campaign_name'] == 'Diwali']
diwali_stores = diwali.groupby(['store_id', 'city'])[quantity_after].sum().sort_values(ascending=False)
print('Answer:', diwali_stores.index[0], diwali_stores.iloc[0])
diwali_stores.head()

## Question 7 - Which campaign had the greater increase in units?

In [ ]:
campaign_totals = data.groupby('campaign_name')[[quantity_before, quantity_after]].sum()
campaign_totals['sales_increase'] = campaign_totals[quantity_after] - campaign_totals[quantity_before]
best_campaign = campaign_totals['sales_increase'].idxmax()
print('Answer:', best_campaign, campaign_totals.loc[best_campaign, 'sales_increase'])
campaign_totals

## Question 8 - Which Sankranti product had the highest IR%?

In [ ]:
sankranti = data[data['campaign_name'] == 'Sankranti']
sankranti_products = calculate_metrics(sankranti, ['product_code', 'product_name'])
sankranti_products = sankranti_products.sort_values('IR%', ascending=False)
top_product = sankranti_products.iloc[0]
print('Answer:', top_product['product_name'], round(top_product['IR%'], 2))
sankranti_products[['product_name', 'IR%']].head()

## Question 9 - Which Visakhapatnam store had the lowest Diwali ISU%?

In [ ]:
vizag_diwali = data[(data['city'] == 'Visakhapatnam') & (data['campaign_name'] == 'Diwali')]
vizag_stores = calculate_metrics(vizag_diwali, ['store_id']).sort_values('ISU%')
lowest_store = vizag_stores.iloc[0]
print('Answer:', lowest_store['store_id'], round(lowest_store['ISU%'], 2))
vizag_stores[['store_id', 'ISU%']]

## Question 10 - Which Sankranti promo had both negative IR% and ISU%?

In [ ]:
promo_metrics = calculate_metrics(sankranti, ['promo_type'])
negative_promo = promo_metrics[(promo_metrics['IR%'] < 0) & (promo_metrics['ISU%'] < 0)]
answer = negative_promo.iloc[0]
print('Answer:', answer['promo_type'], round(answer['IR%'], 2), round(answer['ISU%'], 2))
promo_metrics[['promo_type', 'IR%', 'ISU%']].sort_values('promo_type')

## Final answers

1. **10** duplicate rows
2. **3** cities: Bengaluru, Chennai, Hyderabad
3. **20,78**
4. **Personal Care, 102.38**
5. **34461**
6. **STCHE-4, 5013**
7. **Sankranti, 154175**
8. **Atliq_Suflower_Oil (1L), 91.83**
9. **STVSK-3, 49.21**
10. **25% OFF, -39.33, -19.60**